# 02. 데이터 전처리 (Preprocessing)

## 1. 이 노트북의 목적 및 원칙

`01_data_understanding.ipynb`에서 다음 사항을 이미 실제 데이터로 확인했다.

- `sales`(연도별 5개), `stores`(연도별 5개), `population`(분기별 20개) 파일의 컬럼 구조와 인코딩(`cp949`)
- 세 데이터 모두 2021Q1(20211)~2025Q4(20254) 전체 20개 분기를 누락 없이 포함
- `sales`/`stores`의 grain은 `기준년분기 + 상권코드 + 서비스업종`, `population`의 grain은 `기준년분기 + 상권코드`이며 각각 중복 없음
- `sales`의 key는 `stores`에 100% 포함됨 (join 가능성 확인)
- `stores_2025.csv`만 컬럼명이 영문 코드 체계로 되어 있어 이후 정규화가 필요함
- 황학 관련 4개 상권코드(3110055, 3110057, 3130054, 3130055)가 세 데이터 모두에서 20개 분기 내내 존재함

이번 `02_preprocessing.ipynb`에서는 위에서 확인된 구조를 바탕으로 **실제 분석에 사용할 수 있도록 스키마를 정규화하고 연도/분기별 원본 파일을 통합**한다. `data/raw`의 원본 파일은 어떤 방식으로도 수정하지 않으며, 모든 변환은 메모리상의 DataFrame에서만 수행하고 최종 결과만 `data/processed`에 저장한다.

**이번 노트북에서 하지 않는 것**: 그래프 기반 EDA, 매출 증가/감소나 상권 쇠퇴·회복에 대한 해석, feature engineering(야간매출비율·2030매출비율·상권 활력도·쇠퇴지수·회복지수 등 파생변수 생성), 군집분석·이상탐지 등 모델링, AI/LLM/MCP 연동, Tableau 집계 테이블 생성. 이들은 모두 이후 노트북(`03_eda.ipynb` 이후)에서 다룬다.

**중요 원칙**: 이번 단계에서는 **황학동 4개 상권으로 데이터를 필터링하지 않는다.** 서울 전체 상권 데이터를 그대로 유지해야 이후 (1) 황학동과 서울 전체 상권 비교, (2) 유사 상권 탐색, (3) 서울 전체 상권 군집분석, (4) Tableau 대시보드의 지역/기간/업종 필터링, (5) AI 기반 상권 진단 등이 가능하다. 황학 4개 상권에 대한 검증(섹션 16)은 어디까지나 전처리 과정에서 핵심 분석 대상 데이터가 손실되지 않았는지 확인하기 위한 것이며, 최종 저장 데이터는 서울 전체 상권을 포함한다.


## 2. 라이브러리 및 경로 설정

이 노트북은 `notebooks/` 폴더에 위치하므로, `data/raw`와 `data/processed`는 상위 폴더 기준 상대경로로 접근한다. 절대경로는 사용하지 않는다.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

SALES_DIR = RAW_DIR / "sales"
STORES_DIR = RAW_DIR / "stores"
POPULATION_DIR = RAW_DIR / "population"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name, d in [
    ("SALES_DIR", SALES_DIR),
    ("STORES_DIR", STORES_DIR),
    ("POPULATION_DIR", POPULATION_DIR),
    ("PROCESSED_DIR", PROCESSED_DIR),
]:
    print(f"{name:14s} {d} -> exists: {d.exists()}")


SALES_DIR      ..\data\raw\sales -> exists: True
STORES_DIR     ..\data\raw\stores -> exists: True
POPULATION_DIR ..\data\raw\population -> exists: True
PROCESSED_DIR  ..\data\processed -> exists: True


## 3. 공통 CSV 로딩 함수

`01_data_understanding.ipynb`에서 모든 원본 CSV가 `cp949` 인코딩으로 정상적으로 읽히는 것을 이미 확인했으므로, 이번 노트북에서는 매번 여러 인코딩을 시도하지 않고 `cp949`를 바로 사용한다. 다만 예상과 다르게 오류가 발생하면 원인을 바로 알 수 있도록 예외 메시지를 명확히 남긴다.


In [2]:
def read_csv(path, **kwargs):
    """01에서 확인된 cp949 인코딩으로 CSV를 읽는다. 실패 시 원인을 알 수 있도록 예외를 다시 발생시킨다."""
    try:
        return pd.read_csv(path, encoding="cp949", **kwargs)
    except UnicodeDecodeError as e:
        raise RuntimeError(f"cp949 인코딩으로 읽기 실패: {path} -> {e}")


sales_files = sorted(SALES_DIR.glob("*.csv"))
stores_files = sorted(STORES_DIR.glob("*.csv"))
population_files = sorted(POPULATION_DIR.glob("*.csv"))

print(f"[sales] 파일 수: {len(sales_files)}")
print(f"[stores] 파일 수: {len(stores_files)}")
print(f"[population] 파일 수: {len(population_files)}")

assert len(sales_files) == 5, "sales 원본 파일 수가 예상(5개)과 다릅니다."
assert len(stores_files) == 5, "stores 원본 파일 수가 예상(5개)과 다릅니다."
assert len(population_files) == 20, "population 원본 파일 수가 예상(20개)과 다릅니다."


[sales] 파일 수: 5
[stores] 파일 수: 5
[population] 파일 수: 20


## 4. stores_2025 컬럼명 정규화

`01`에서 확인한 것처럼 `stores_2021~2024.csv`는 한글 컬럼명을 쓰지만 `stores_2025.csv`만 동일한 의미의 영문 코드 컬럼명(`stdr_yyqu_cd` 등)을 사용한다. 이후 5개 파일을 하나로 합치려면 반드시 컬럼명을 통일해야 하므로, **컬럼 위치가 아닌 명시적 매핑 dictionary**로 `stores_2025`만 한글 컬럼명으로 rename한다. 위치 기반 매핑은 향후 원본 컬럼 순서가 바뀌면 조용히 잘못된 매핑을 만들 수 있어 위험하다.


In [3]:
STORES_2025_COLUMN_MAP = {
    "stdr_yyqu_cd": "기준_년분기_코드",
    "trdar_se_cd": "상권_구분_코드",
    "trdar_se_cd_nm": "상권_구분_코드_명",
    "trdar_cd": "상권_코드",
    "trdar_cd_nm": "상권_코드_명",
    "svc_induty_cd": "서비스_업종_코드",
    "svc_induty_cd_nm": "서비스_업종_코드_명",
    "stor_co": "점포_수",
    "similr_induty_stor_co": "유사_업종_점포_수",
    "opbiz_rt": "개업_율",
    "opbiz_stor_co": "개업_점포_수",
    "clsbiz_rt": "폐업_률",
    "clsbiz_stor_co": "폐업_점포_수",
    "frc_stor_co": "프랜차이즈_점포_수",
}

stores_2021 = read_csv(STORES_DIR / "stores_2021.csv")
stores_2022 = read_csv(STORES_DIR / "stores_2022.csv")
stores_2023 = read_csv(STORES_DIR / "stores_2023.csv")
stores_2024 = read_csv(STORES_DIR / "stores_2024.csv")
stores_2025_raw = read_csv(STORES_DIR / "stores_2025.csv")

print("stores_2025 원본 컬럼 (rename 전):")
print(stores_2025_raw.columns.tolist())


stores_2025 원본 컬럼 (rename 전):
['stdr_yyqu_cd', 'trdar_se_cd', 'trdar_se_cd_nm', 'trdar_cd', 'trdar_cd_nm', 'svc_induty_cd', 'svc_induty_cd_nm', 'stor_co', 'similr_induty_stor_co', 'opbiz_rt', 'opbiz_stor_co', 'clsbiz_rt', 'clsbiz_stor_co', 'frc_stor_co']


In [4]:
# 매핑 dict의 키(영문 컬럼명)가 실제 stores_2025.csv 컬럼과 정확히 일치하는지 먼저 검증한다.
# 위치(index) 기반이 아니라, 컬럼명 자체를 명시적으로 대조한다.
missing_in_2025 = set(STORES_2025_COLUMN_MAP.keys()) - set(stores_2025_raw.columns)
extra_in_2025 = set(stores_2025_raw.columns) - set(STORES_2025_COLUMN_MAP.keys())

assert not missing_in_2025, f"매핑 dict에는 있지만 실제 stores_2025.csv에는 없는 컬럼: {missing_in_2025}"
assert not extra_in_2025, f"실제 stores_2025.csv에는 있지만 매핑 dict에는 없는 컬럼: {extra_in_2025}"
print("stores_2025.csv 컬럼과 STORES_2025_COLUMN_MAP 키가 완전히 일치함을 확인했습니다.")

# 원본 stores_2025_raw는 그대로 두고, rename된 새 DataFrame(메모리상)만 만든다.
stores_2025 = stores_2025_raw.rename(columns=STORES_2025_COLUMN_MAP)

print("\nrename 후 stores_2025 컬럼:")
print(stores_2025.columns.tolist())


stores_2025.csv 컬럼과 STORES_2025_COLUMN_MAP 키가 완전히 일치함을 확인했습니다.

rename 후 stores_2025 컬럼:
['기준_년분기_코드', '상권_구분_코드', '상권_구분_코드_명', '상권_코드', '상권_코드_명', '서비스_업종_코드', '서비스_업종_코드_명', '점포_수', '유사_업종_점포_수', '개업_율', '개업_점포_수', '폐업_률', '폐업_점포_수', '프랜차이즈_점포_수']


In [5]:
# 2021~2025 stores의 컬럼 개수 / 이름 / 순서가 완전히 동일한지 확인한다.
# 동일하지 않다면 곧바로 concat하지 않고 차이를 출력해서 원인을 먼저 확인한다.
stores_year_frames = {
    "2021": stores_2021,
    "2022": stores_2022,
    "2023": stores_2023,
    "2024": stores_2024,
    "2025": stores_2025,
}

for label, df in stores_year_frames.items():
    print(f"stores_{label} 컬럼 수: {len(df.columns)}")

base_cols = stores_2021.columns.tolist()
cols_match = all(df.columns.tolist() == base_cols for df in stores_year_frames.values())
print("\n2021~2025 stores 컬럼(개수+이름+순서) 완전 일치:", cols_match)

if not cols_match:
    for label, df in stores_year_frames.items():
        if df.columns.tolist() != base_cols:
            print(f"stores_{label} 차이(대칭차집합): {set(df.columns) ^ set(base_cols)}")
            print(f"stores_{label} 실제 컬럼 순서: {df.columns.tolist()}")
    raise AssertionError("stores 컬럼 구조가 완전히 일치하지 않습니다. 원인을 먼저 확인한 뒤 진행해야 합니다.")

print("\nstores_2021.columns.tolist() == stores_2025.columns.tolist():",
      stores_2021.columns.tolist() == stores_2025.columns.tolist())


stores_2021 컬럼 수: 14
stores_2022 컬럼 수: 14
stores_2023 컬럼 수: 14
stores_2024 컬럼 수: 14
stores_2025 컬럼 수: 14

2021~2025 stores 컬럼(개수+이름+순서) 완전 일치: True

stores_2021.columns.tolist() == stores_2025.columns.tolist(): True


## 5. sales 2021~2025 통합

`sales`는 5개 파일 모두 원래부터 동일한 55개 한글 컬럼 구조를 사용하므로(01 확인), 컬럼 구조를 마지막으로 한 번 더 확인한 뒤 그대로 `concat`한다.


In [6]:
sales_frames = {f.stem: read_csv(f) for f in sales_files}

base_sales_cols = sales_frames[sales_files[0].stem].columns.tolist()
for name, df in sales_frames.items():
    assert df.columns.tolist() == base_sales_cols, (
        f"{name} 컬럼 구조가 기준과 다릅니다: {set(df.columns) ^ set(base_sales_cols)}"
    )
print("sales 5개 파일 컬럼 구조(개수/이름/순서) 완전 일치 확인 완료 (컬럼 수:", len(base_sales_cols), ")")

sales_row_counts = {name: len(df) for name, df in sales_frames.items()}
print("\n파일별 행 수:", sales_row_counts)

sales = pd.concat(list(sales_frames.values()), ignore_index=True)

assert sum(sales_row_counts.values()) == len(sales), "concat 전/후 행 수가 일치하지 않습니다."
print("concat 전/후 행 수 일치 확인:", sum(sales_row_counts.values()), "==", len(sales))


sales 5개 파일 컬럼 구조(개수/이름/순서) 완전 일치 확인 완료 (컬럼 수: 55 )

파일별 행 수: {'sales_2021': 89150, 'sales_2022': 88834, 'sales_2023': 88246, 'sales_2024': 87179, 'sales_2025': 85732}
concat 전/후 행 수 일치 확인: 439141 == 439141


In [7]:
print("[sales 통합 결과]")
print("shape:", sales.shape)
print("기간 min/max:", sales["기준_년분기_코드"].min(), "~", sales["기준_년분기_코드"].max())
print("고유 분기 수:", sales["기준_년분기_코드"].nunique())
print("상권 수:", sales["상권_코드"].nunique())
print("서비스 업종 수:", sales["서비스_업종_코드"].nunique())
sales.head(3)


[sales 통합 결과]
shape: (439141, 55)
기간 min/max: 20211 ~ 20254
고유 분기 수: 20
상권 수: 1596
서비스 업종 수: 63


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,월요일_매출_금액,화요일_매출_금액,수요일_매출_금액,목요일_매출_금액,금요일_매출_금액,토요일_매출_금액,일요일_매출_금액,시간대_00~06_매출_금액,시간대_06~11_매출_금액,시간대_11~14_매출_금액,시간대_14~17_매출_금액,시간대_17~21_매출_금액,시간대_21~24_매출_금액,남성_매출_금액,여성_매출_금액,연령대_10_매출_금액,연령대_20_매출_금액,연령대_30_매출_금액,연령대_40_매출_금액,연령대_50_매출_금액,연령대_60_이상_매출_금액,주중_매출_건수,주말_매출_건수,월요일_매출_건수,화요일_매출_건수,수요일_매출_건수,목요일_매출_건수,금요일_매출_건수,토요일_매출_건수,일요일_매출_건수,시간대_건수~06_매출_건수,시간대_건수~11_매출_건수,시간대_건수~14_매출_건수,시간대_건수~17_매출_건수,시간대_건수~21_매출_건수,시간대_건수~24_매출_건수,남성_매출_건수,여성_매출_건수,연령대_10_매출_건수,연령대_20_매출_건수,연령대_30_매출_건수,연령대_40_매출_건수,연령대_50_매출_건수,연령대_60_이상_매출_건수
0,20211,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,382839289,13701,235308851,147530438,35385544,53566081,53687402,47097897,45571927,70698283,76832155,0,2699877,138275545,77483113,157839930,6540824,218578196,102157686,888451,12720922,26575259,59039301,111451438,110060512,9487,4214,1686,2059,2170,1793,1779,2286,1928,0,128,6211,2701,4520,141,8000,4228,102,826,1174,2467,4049,3615
1,20211,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,122957138,12039,61070094,61887044,8611528,15264108,10629354,15464848,11100256,32014369,29872675,11373269,37723773,43587908,26677599,3594589,0,64010667,47678846,251721,4661073,10112557,25115187,48256159,23292816,6435,5604,1120,1319,1463,1164,1369,2809,2795,289,4245,5060,1893,552,0,6679,4293,52,926,1222,2698,3404,2670
2,20211,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,19012827,264,14322928,4689899,1355810,4118094,5868849,726356,2253819,1217565,3472334,0,0,0,460616,9640869,8911342,11667789,3401471,0,0,611202,575770,8576819,5305469,174,90,23,28,52,27,44,37,53,0,0,0,18,188,58,182,45,0,0,9,18,102,97


## 6. stores 2021~2025 통합

`stores_2025`는 앞서 정규화한 컬럼명을 사용하므로, 5개 연도 파일을 그대로 `concat`할 수 있다.


In [8]:
stores_row_counts = {label: len(df) for label, df in stores_year_frames.items()}
print("파일별 행 수:", stores_row_counts)

stores = pd.concat(list(stores_year_frames.values()), ignore_index=True)

assert sum(stores_row_counts.values()) == len(stores), "concat 전/후 행 수가 일치하지 않습니다."
print("concat 전/후 행 수 일치 확인:", sum(stores_row_counts.values()), "==", len(stores))


파일별 행 수: {'2021': 303880, '2022': 305587, '2023': 307741, '2024': 306889, '2025': 304775}
concat 전/후 행 수 일치 확인: 1528872 == 1528872


In [9]:
print("[stores 통합 결과]")
print("shape:", stores.shape)
print("기간 min/max:", stores["기준_년분기_코드"].min(), "~", stores["기준_년분기_코드"].max())
print("고유 분기 수:", stores["기준_년분기_코드"].nunique())
print("상권 수:", stores["상권_코드"].nunique())
print("서비스 업종 수:", stores["서비스_업종_코드"].nunique())
stores.head(3)


[stores 통합 결과]
shape: (1528872, 14)
기간 min/max: 20211 ~ 20254
고유 분기 수: 20
상권 수: 1650
서비스 업종 수: 100


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,점포_수,유사_업종_점포_수,개업_율,개업_점포_수,폐업_률,폐업_점포_수,프랜차이즈_점포_수
0,20211,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,12,0,0,0,0,0
1,20211,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,3,3,0,0,0,0,0
2,20211,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,3,4,0,0,0,0,1


## 7. population 2021Q1~2025Q4 통합

`population`은 파일명 자체에 분기가 표시된다(`population_20211.csv` 등). 통합하기 전에 파일명이 나타내는 분기와 파일 내부 `기준_년분기_코드` 실제 값이 서로 일치하는지 다시 확인한다. 파일명과 내용이 어긋나면 이후 시계열 분석에서 엉뚱한 분기로 데이터를 잘못 라벨링하게 되므로 반드시 짚고 넘어가야 한다.


In [10]:
population_frames = {f.stem: read_csv(f) for f in population_files}

base_population_cols = population_frames[population_files[0].stem].columns.tolist()
for name, df in population_frames.items():
    assert df.columns.tolist() == base_population_cols, f"{name} 컬럼 구조가 기준과 다릅니다."
print("population 20개 파일 컬럼 구조 완전 일치 확인 완료 (컬럼 수:", len(base_population_cols), ")")


population 20개 파일 컬럼 구조 완전 일치 확인 완료 (컬럼 수: 27 )


In [11]:
print("[population] 파일명 분기 vs 데이터 내부 기준_년분기_코드 일치 확인")
mismatched_files = []
for f in population_files:
    expected = f.stem.replace("population_", "")
    actual = sorted(population_frames[f.stem]["기준_년분기_코드"].unique().tolist())
    is_match = (len(actual) == 1) and (str(actual[0]) == expected)
    status = "OK" if is_match else "확인 필요"
    print(f"  - {f.name}: 파일명 기준={expected}, 데이터 내부 값={actual} -> {status}")
    if not is_match:
        mismatched_files.append(f.name)

assert not mismatched_files, f"파일명과 내부 기준년분기가 불일치하는 파일이 있습니다: {mismatched_files}"
print("\n불일치 파일: 없음")


[population] 파일명 분기 vs 데이터 내부 기준_년분기_코드 일치 확인
  - population_20211.csv: 파일명 기준=20211, 데이터 내부 값=[20211] -> OK
  - population_20212.csv: 파일명 기준=20212, 데이터 내부 값=[20212] -> OK
  - population_20213.csv: 파일명 기준=20213, 데이터 내부 값=[20213] -> OK
  - population_20214.csv: 파일명 기준=20214, 데이터 내부 값=[20214] -> OK
  - population_20221.csv: 파일명 기준=20221, 데이터 내부 값=[20221] -> OK
  - population_20222.csv: 파일명 기준=20222, 데이터 내부 값=[20222] -> OK
  - population_20223.csv: 파일명 기준=20223, 데이터 내부 값=[20223] -> OK
  - population_20224.csv: 파일명 기준=20224, 데이터 내부 값=[20224] -> OK
  - population_20231.csv: 파일명 기준=20231, 데이터 내부 값=[20231] -> OK
  - population_20232.csv: 파일명 기준=20232, 데이터 내부 값=[20232] -> OK
  - population_20233.csv: 파일명 기준=20233, 데이터 내부 값=[20233] -> OK
  - population_20234.csv: 파일명 기준=20234, 데이터 내부 값=[20234] -> OK
  - population_20241.csv: 파일명 기준=20241, 데이터 내부 값=[20241] -> OK
  - population_20242.csv: 파일명 기준=20242, 데이터 내부 값=[20242] -> OK
  - population_20243.csv: 파일명 기준=20243, 데이터 내부 값=[20243] -> OK
  - popul

In [12]:
population_row_counts = {label: len(df) for label, df in population_frames.items()}

population = pd.concat(list(population_frames.values()), ignore_index=True)

assert sum(population_row_counts.values()) == len(population), "concat 전/후 행 수가 일치하지 않습니다."
print("concat 전/후 행 수 일치 확인:", sum(population_row_counts.values()), "==", len(population))

print("\n[population 통합 결과]")
print("shape:", population.shape)
print("기간 min/max:", population["기준_년분기_코드"].min(), "~", population["기준_년분기_코드"].max())
print("고유 분기 수:", population["기준_년분기_코드"].nunique())
print("상권 수:", population["상권_코드"].nunique())
population.head(3)


concat 전/후 행 수 일치 확인: 32984 == 32984

[population 통합 결과]
shape: (32984, 27)
기간 min/max: 20211 ~ 20254
고유 분기 수: 20
상권 수: 1650


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,총_유동인구_수,남성_유동인구_수,여성_유동인구_수,연령대_10_유동인구_수,연령대_20_유동인구_수,연령대_30_유동인구_수,연령대_40_유동인구_수,연령대_50_유동인구_수,연령대_60_이상_유동인구_수,시간대_00_06_유동인구_수,시간대_06_11_유동인구_수,시간대_11_14_유동인구_수,시간대_14_17_유동인구_수,시간대_17_21_유동인구_수,시간대_21_24_유동인구_수,월요일_유동인구_수,화요일_유동인구_수,수요일_유동인구_수,목요일_유동인구_수,금요일_유동인구_수,토요일_유동인구_수,일요일_유동인구_수
0,20211,R,전통시장,3130093,동서시장,55869,28962,26906,2383,4236,5347,7525,10818,25561,7701,11527,10785,11447,9583,4824,8392,8465,8576,8219,7926,7760,6530
1,20211,A,골목상권,3111004,삼전역 1번,2228673,1035991,1192681,346754,341951,450172,381968,285522,422307,627736,465543,245684,239313,352222,298173,319258,318087,318491,317386,314453,321281,319715
2,20211,A,골목상권,3111005,삼전역 3번,1297239,597924,699315,194503,185054,279049,213435,181337,243861,430080,279578,124080,110097,171886,181519,186155,182724,182943,182740,182483,186612,193581


## 8. 컬럼명 품질 점검

통합된 세 데이터의 컬럼명에 앞뒤 공백, 중복 컬럼명, 의도하지 않은 공백 등 명백한 문제가 없는지 점검한다. 컬럼명 전체를 영어 snake_case로 바꾸는 등 임의 변경은 하지 않고, 현재의 한글 컬럼명 체계를 그대로 유지한다.


In [13]:
def check_column_quality(df, label):
    cols = df.columns.tolist()
    issues = []

    stripped_mismatch = [c for c in cols if c != c.strip()]
    if stripped_mismatch:
        issues.append(f"앞뒤 공백이 있는 컬럼: {stripped_mismatch}")

    dup_cols = pd.Series(cols)[pd.Series(cols).duplicated()].tolist()
    if dup_cols:
        issues.append(f"중복 컬럼명: {dup_cols}")

    weird_cols = [c for c in cols if "  " in c or c == ""]
    if weird_cols:
        issues.append(f"의도하지 않은 공백/빈 컬럼명 의심: {weird_cols}")

    print(f"[{label}] 컬럼 품질 이슈:", issues if issues else "없음")


check_column_quality(sales, "sales")
check_column_quality(stores, "stores")
check_column_quality(population, "population")


[sales] 컬럼 품질 이슈: 없음
[stores] 컬럼 품질 이슈: 없음
[population] 컬럼 품질 이슈: 없음


### 8-1. sales 시간대별 매출 건수 컬럼명 확인

`01`에서 sales의 시간대별 매출 **건수** 컬럼명이 다음과 같이 다소 비정상적으로 보이는 것을 확인했다.

```
시간대_건수~06_매출_건수, 시간대_건수~11_매출_건수, 시간대_건수~14_매출_건수,
시간대_건수~17_매출_건수, 시간대_건수~21_매출_건수, 시간대_건수~24_매출_건수
```

같은 구조의 시간대별 매출 **금액** 컬럼은 `시간대_00~06_매출_금액` ~ `시간대_21~24_매출_금액`로 정상적인 이름을 쓰고 있고, 두 그룹은 데이터 내에서 완전히 동일한 순서(00~06 → 06~11 → 11~14 → 14~17 → 17~21 → 21~24)로 나열되어 있다. 즉 시간대 구간의 앞자리 숫자("00", "06", "11", ...)가 "건수"라는 글자로 잘못 치환된 것으로 보이는 표기 오류이며, 6개 시간대 구간에 정확히 대응된다는 것이 컬럼 순서와 명명 패턴상 명확하므로 명시적 매핑으로 정리한다.


In [14]:
amount_time_cols = [c for c in sales.columns if c.startswith("시간대_") and c.endswith("_매출_금액")]
count_time_cols_before = [c for c in sales.columns if "시간대" in c and c.endswith("_매출_건수")]

print("시간대별 매출 금액 컬럼(정상 표기, 참고용):", amount_time_cols)
print("시간대별 매출 건수 컬럼(rename 전, 비정상 표기):", count_time_cols_before)


시간대별 매출 금액 컬럼(정상 표기, 참고용): ['시간대_00~06_매출_금액', '시간대_06~11_매출_금액', '시간대_11~14_매출_금액', '시간대_14~17_매출_금액', '시간대_17~21_매출_금액', '시간대_21~24_매출_금액']
시간대별 매출 건수 컬럼(rename 전, 비정상 표기): ['시간대_건수~06_매출_건수', '시간대_건수~11_매출_건수', '시간대_건수~14_매출_건수', '시간대_건수~17_매출_건수', '시간대_건수~21_매출_건수', '시간대_건수~24_매출_건수']


In [15]:
TIME_COUNT_RENAME_MAP = {
    "시간대_건수~06_매출_건수": "시간대_00~06_매출_건수",
    "시간대_건수~11_매출_건수": "시간대_06~11_매출_건수",
    "시간대_건수~14_매출_건수": "시간대_11~14_매출_건수",
    "시간대_건수~17_매출_건수": "시간대_14~17_매출_건수",
    "시간대_건수~21_매출_건수": "시간대_17~21_매출_건수",
    "시간대_건수~24_매출_건수": "시간대_21~24_매출_건수",
}

missing_targets = set(TIME_COUNT_RENAME_MAP.keys()) - set(sales.columns)
assert not missing_targets, f"rename 대상 컬럼이 sales에 존재하지 않습니다: {missing_targets}"

n_cols_before = len(sales.columns)
sales = sales.rename(columns=TIME_COUNT_RENAME_MAP)
n_cols_after = len(sales.columns)

assert n_cols_before == n_cols_after, "rename 전후 컬럼 수가 달라졌습니다."
assert sales.columns.duplicated().sum() == 0, "rename으로 인해 중복 컬럼이 새로 생성되었습니다."

print("rename 완료. 컬럼 수:", n_cols_before, "->", n_cols_after, "(중복 컬럼 없음 확인)")
print("변경된 컬럼명:", list(TIME_COUNT_RENAME_MAP.values()))


rename 완료. 컬럼 수: 55 -> 55 (중복 컬럼 없음 확인)
변경된 컬럼명: ['시간대_00~06_매출_건수', '시간대_06~11_매출_건수', '시간대_11~14_매출_건수', '시간대_14~17_매출_건수', '시간대_17~21_매출_건수', '시간대_21~24_매출_건수']


## 9. 데이터 타입 정리

향후 Tableau 필터링, 서울 전체 상권 비교, AI 기반 질의에서 상권코드·업종코드 등 식별자 값이 계산 과정에서 변형되면 안 되므로, **코드형 식별자는 계산용 수치가 아니라 식별자**라는 원칙을 적용한다. 특히 `상권_구분_코드`(A/D/R/U 등 알파벳)와 `서비스_업종_코드`(`CS100001` 등 문자+숫자 조합)는 애초에 숫자로만 구성되어 있지 않아 파일을 읽을 때부터 문자열 계열로 처리되지만, 이를 가정하지 않고 실제 값과 dtype을 확인한 뒤 필요한 경우에만 명시적으로 정리한다.


In [16]:
KEY_TEXT_COLS = ["상권_구분_코드", "상권_구분_코드_명", "상권_코드_명", "서비스_업종_코드", "서비스_업종_코드_명"]

print("[변경 전 핵심 key 컬럼 dtype]")
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    cols = [c for c in ["기준_년분기_코드", "상권_구분_코드", "상권_구분_코드_명", "상권_코드", "상권_코드_명",
                          "서비스_업종_코드", "서비스_업종_코드_명"] if c in df.columns]
    print(f"\n[{label}]")
    print(df[cols].dtypes)


[변경 전 핵심 key 컬럼 dtype]

[sales]
기준_년분기_코드      int64
상권_구분_코드         str
상권_구분_코드_명       str
상권_코드          int64
상권_코드_명          str
서비스_업종_코드        str
서비스_업종_코드_명      str
dtype: object

[stores]
기준_년분기_코드      int64
상권_구분_코드         str
상권_구분_코드_명       str
상권_코드          int64
상권_코드_명          str
서비스_업종_코드        str
서비스_업종_코드_명      str
dtype: object

[population]
기준_년분기_코드     int64
상권_구분_코드        str
상권_구분_코드_명      str
상권_코드         int64
상권_코드_명         str
dtype: object


In [17]:
# 상권_구분_코드가 실제로 숫자가 아닌 값(A/D/R/U 등)만 가지는지 확인한다.
# 만약 전부 숫자로만 구성된 코드라면 int로 캐스팅될 때 앞자리 0이 손실될 위험이 있으므로,
# 먼저 실제 값을 눈으로 확인한 뒤 dtype 정책을 결정한다.
print("상권_구분_코드 고유값 (sales):", sorted(sales["상권_구분_코드"].unique().tolist()))
print("서비스_업종_코드 예시 (sales):", sales["서비스_업종_코드"].unique()[:5].tolist())
print("서비스_업종_코드 전체가 'CS'로 시작하는가:", sales["서비스_업종_코드"].str.startswith("CS").all())


상권_구분_코드 고유값 (sales): ['A', 'D', 'R', 'U']
서비스_업종_코드 예시 (sales): ['CS100001', 'CS100008', 'CS100009', 'CS200001', 'CS300002']
서비스_업종_코드 전체가 'CS'로 시작하는가: True


In [18]:
def to_int_safely(df, col, label):
    """기준_년분기_코드, 상권_코드처럼 순수 숫자 식별자를 int64로 명시 캐스팅하되,
    캐스팅 전후로 고유값 집합과 결측치 수가 동일한지 검증하여 정보 손실이 없는지 확인한다."""
    before_unique = set(df[col].unique().tolist())
    before_na = int(df[col].isna().sum())

    df[col] = df[col].astype("int64")

    after_unique = set(df[col].unique().tolist())
    after_na = int(df[col].isna().sum())

    assert before_unique == after_unique, f"{label}.{col}: int64 변환 후 고유값 집합이 달라졌습니다."
    assert before_na == after_na, f"{label}.{col}: int64 변환 후 결측치 수가 달라졌습니다."
    print(f"[{label}.{col}] int64 확인/변환 완료 (고유값 {len(after_unique)}개, 결측 {after_na}개, 값 변화 없음)")


for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    to_int_safely(df, "기준_년분기_코드", label)
    to_int_safely(df, "상권_코드", label)


[sales.기준_년분기_코드] int64 확인/변환 완료 (고유값 20개, 결측 0개, 값 변화 없음)
[sales.상권_코드] int64 확인/변환 완료 (고유값 1596개, 결측 0개, 값 변화 없음)
[stores.기준_년분기_코드] int64 확인/변환 완료 (고유값 20개, 결측 0개, 값 변화 없음)
[stores.상권_코드] int64 확인/변환 완료 (고유값 1650개, 결측 0개, 값 변화 없음)
[population.기준_년분기_코드] int64 확인/변환 완료 (고유값 20개, 결측 0개, 값 변화 없음)
[population.상권_코드] int64 확인/변환 완료 (고유값 1650개, 결측 0개, 값 변화 없음)


In [19]:
def ensure_string_dtype(df, col, label):
    """코드/코드명 컬럼이 문자열 계열 dtype인지 확인하고, 아니라면 str로 캐스팅한다.
    캐스팅 전후 고유값 개수와 결측치 수, 대표값이 그대로인지 검증한다 (앞자리 0 등 정보 손실 방지)."""
    if col not in df.columns:
        return
    before_unique_n = df[col].nunique(dropna=True)
    before_na = int(df[col].isna().sum())
    before_sample = df[col].dropna().iloc[0]

    if not pd.api.types.is_string_dtype(df[col]):
        df[col] = df[col].astype(str)

    after_unique_n = df[col].nunique(dropna=True)
    after_na = int(df[col].isna().sum())
    after_sample = df[col].dropna().iloc[0]

    assert before_unique_n == after_unique_n, f"{label}.{col}: 문자열 dtype 정리 후 고유값 개수가 달라졌습니다."
    assert before_na == after_na, f"{label}.{col}: 문자열 dtype 정리 후 결측치 수가 달라졌습니다."
    print(f"[{label}.{col}] dtype={df[col].dtype}, 고유값 {after_unique_n}개, 결측 {after_na}개, "
          f"예시 '{before_sample}' -> '{after_sample}' (값 변화 없음)")


for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    print(f"\n[{label}]")
    for col in KEY_TEXT_COLS:
        ensure_string_dtype(df, col, label)



[sales]
[sales.상권_구분_코드] dtype=str, 고유값 4개, 결측 0개, 예시 'A' -> 'A' (값 변화 없음)
[sales.상권_구분_코드_명] dtype=str, 고유값 4개, 결측 0개, 예시 '골목상권' -> '골목상권' (값 변화 없음)
[sales.상권_코드_명] dtype=str, 고유값 1597개, 결측 0개, 예시 '이북5도청사' -> '이북5도청사' (값 변화 없음)
[sales.서비스_업종_코드] dtype=str, 고유값 63개, 결측 0개, 예시 'CS100001' -> 'CS100001' (값 변화 없음)
[sales.서비스_업종_코드_명] dtype=str, 고유값 63개, 결측 0개, 예시 '한식음식점' -> '한식음식점' (값 변화 없음)

[stores]
[stores.상권_구분_코드] dtype=str, 고유값 4개, 결측 0개, 예시 'A' -> 'A' (값 변화 없음)


[stores.상권_구분_코드_명] dtype=str, 고유값 4개, 결측 0개, 예시 '골목상권' -> '골목상권' (값 변화 없음)


[stores.상권_코드_명] dtype=str, 고유값 1651개, 결측 0개, 예시 '이북5도청사' -> '이북5도청사' (값 변화 없음)


[stores.서비스_업종_코드] dtype=str, 고유값 100개, 결측 0개, 예시 'CS100001' -> 'CS100001' (값 변화 없음)


[stores.서비스_업종_코드_명] dtype=str, 고유값 100개, 결측 0개, 예시 '한식음식점' -> '한식음식점' (값 변화 없음)

[population]
[population.상권_구분_코드] dtype=str, 고유값 4개, 결측 0개, 예시 'R' -> 'R' (값 변화 없음)
[population.상권_구분_코드_명] dtype=str, 고유값 4개, 결측 0개, 예시 '전통시장' -> '전통시장' (값 변화 없음)
[population.상권_코드_명] dtype=str, 고유값 1652개, 결측 0개, 예시 '동서시장' -> '동서시장' (값 변화 없음)


In [20]:
assert sales["상권_코드"].dtype == stores["상권_코드"].dtype == population["상권_코드"].dtype, \
    "세 데이터의 상권_코드 dtype이 서로 다릅니다."
assert sales["기준_년분기_코드"].dtype == stores["기준_년분기_코드"].dtype == population["기준_년분기_코드"].dtype, \
    "세 데이터의 기준_년분기_코드 dtype이 서로 다릅니다."
assert sales["서비스_업종_코드"].dtype == stores["서비스_업종_코드"].dtype, \
    "sales/stores의 서비스_업종_코드 dtype이 서로 다릅니다."

print("핵심 key 컬럼 dtype 일치 확인 완료")
print("- 상권_코드 dtype       :", sales["상권_코드"].dtype)
print("- 기준_년분기_코드 dtype:", sales["기준_년분기_코드"].dtype)
print("- 서비스_업종_코드 dtype:", sales["서비스_업종_코드"].dtype)


핵심 key 컬럼 dtype 일치 확인 완료
- 상권_코드 dtype       : int64
- 기준_년분기_코드 dtype: int64
- 서비스_업종_코드 dtype: str


## 10. 분석 key 무결성 검증

Tableau 필터링, 유사 상권 비교, AI 분석 모두 기간·상권·업종 식별자가 정확하다는 전제 위에서 동작한다. 여기서는 (1) `기준_년분기_코드`가 정확히 20211~20254 범위 안에 있는지, (2) 동일한 상권코드/업종코드가 서로 다른 이름과 연결되는 사례가 있는지를 확인한다. 문제가 발견되어도 이 단계에서 임의로 수정하지 않고 사실만 기록한다.


In [21]:
expected_quarters = [int(f"{y}{q}") for y in range(2021, 2026) for q in range(1, 5)]
print("기대되는 20211~20254 전체 분기 수:", len(expected_quarters))

for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    actual_quarters = set(df["기준_년분기_코드"].unique().tolist())
    unexpected = actual_quarters - set(expected_quarters)
    missing = set(expected_quarters) - actual_quarters
    print(f"[{label}] 예상 범위를 벗어난 값: {unexpected if unexpected else '없음'} / "
          f"누락된 분기: {sorted(missing) if missing else '없음'}")
    assert not unexpected, f"{label}에 20211~20254 범위를 벗어난 기준_년분기_코드가 있습니다: {unexpected}"


기대되는 20211~20254 전체 분기 수: 20
[sales] 예상 범위를 벗어난 값: 없음 / 누락된 분기: 없음
[stores] 예상 범위를 벗어난 값: 없음 / 누락된 분기: 없음
[population] 예상 범위를 벗어난 값: 없음 / 누락된 분기: 없음


In [22]:
# 연도/분기 부분이 각각 논리적으로 유효한 범위인지도 확인한다 (예: 연도 2021~2025, 분기 1~4).
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    years = df["기준_년분기_코드"] // 10
    quarters = df["기준_년분기_코드"] % 10
    valid_year = bool(years.between(2021, 2025).all())
    valid_quarter = bool(quarters.between(1, 4).all())
    print(f"[{label}] 연도부(2021~2025) 유효성: {valid_year} / 분기부(1~4) 유효성: {valid_quarter}")
    assert valid_year and valid_quarter, f"{label}의 기준_년분기_코드 중 연도/분기 부분이 논리적으로 유효하지 않은 값이 있습니다."


[sales] 연도부(2021~2025) 유효성: True / 분기부(1~4) 유효성: True
[stores] 연도부(2021~2025) 유효성: True / 분기부(1~4) 유효성: True
[population] 연도부(2021~2025) 유효성: True / 분기부(1~4) 유효성: True


In [23]:
def check_code_name_consistency(df, code_col, name_col, label):
    """동일 코드가 서로 다른 이름과 연결되는 사례, 그리고 동일 이름이 서로 다른 코드와 연결되는 사례를 각각 확인한다.
    후자는 오류로 단정하지 않고 사례만 출력한다 (동명이지만 실제로 다른 상권일 수 있음)."""
    code_to_name_n = df.groupby(code_col)[name_col].nunique()
    multi_name_codes = code_to_name_n[code_to_name_n > 1]
    print(f"[{label}] 동일 {code_col}가 서로 다른 {name_col}과 연결된 사례: {len(multi_name_codes)}건")
    if len(multi_name_codes) > 0:
        display(df[df[code_col].isin(multi_name_codes.index)][[code_col, name_col]]
                .drop_duplicates().sort_values(code_col).head(20))

    name_to_code_n = df.groupby(name_col)[code_col].nunique()
    multi_code_names = name_to_code_n[name_to_code_n > 1]
    print(f"[{label}] 동일 {name_col}이 서로 다른 {code_col}와 연결된 사례: {len(multi_code_names)}건 "
          f"(오류로 단정하지 않고 사례만 기록)")
    if len(multi_code_names) > 0:
        display(df[df[name_col].isin(multi_code_names.index)][[code_col, name_col]]
                .drop_duplicates().sort_values(name_col).head(20))


print("### 상권_코드 <-> 상권_코드_명")
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    check_code_name_consistency(df, "상권_코드", "상권_코드_명", label)


### 상권_코드 <-> 상권_코드_명
[sales] 동일 상권_코드가 서로 다른 상권_코드_명과 연결된 사례: 1건


,상권_코드,상권_코드_명
132,3110024,혜회동주민센터
353772,3110024,혜화동주민센터


[sales] 동일 상권_코드_명이 서로 다른 상권_코드와 연결된 사례: 0건 (오류로 단정하지 않고 사례만 기록)


[stores] 동일 상권_코드가 서로 다른 상권_코드_명과 연결된 사례: 1건


,상권_코드,상권_코드_명
750,3110024,혜회동주민센터
1224870,3110024,혜화동주민센터


[stores] 동일 상권_코드_명이 서로 다른 상권_코드와 연결된 사례: 0건 (오류로 단정하지 않고 사례만 기록)


[population] 동일 상권_코드가 서로 다른 상권_코드_명과 연결된 사례: 2건


,상권_코드,상권_코드_명
1090,3110024,혜회동주민센터
26732,3110024,혜화동주민센터
1447,3110379,KT&G 북부지사
27086,3110379,KTNG 북부지사


[population] 동일 상권_코드_명이 서로 다른 상권_코드와 연결된 사례: 0건 (오류로 단정하지 않고 사례만 기록)


### 10-1. 상권코드 ↔ 상권명 표기 불일치에 대한 처리 방침

위 결과에서 동일한 `상권_코드`가 서로 다른 `상권_코드_명`과 연결되는 사례가 발견되었다 (예: `상권_코드` 3110024가 sales/stores/population에서 `혜회동주민센터`와 `혜화동주민센터` 두 표기로, population에서는 `상권_코드` 3110379가 `KT&G 북부지사`와 `KTNG 북부지사` 두 표기로 연결됨). 이는 원본 데이터 자체의 표기 편차로 보이며, 현재 시점에는 어느 쪽이 공식 명칭인지 판단할 근거(서울시 공식 상권 정의, `commercial_area` GIS 속성 테이블 등과의 대조)가 없으므로 이번 노트북에서는 표기를 임의로 통일하거나 별도의 canonical-name 컬럼을 만들지 않는다.

이에 따라 향후 분석에서는 다음 원칙을 적용한다.

- **`상권_코드`를 기본 식별자로 사용**하고, `상권_코드_명`은 화면 표시(레이블링)용 속성으로만 취급한다.
- Tableau 등에서도 상권명을 join key처럼 사용하지 않는다. 표기 편차가 있는 이름으로 조인하면 일부 행이 조용히 누락될 수 있다.
- 필요하다면 이후 단계에서 `commercial_area`(GIS) 데이터나 서울시 공식 상권 정의와 대조하여 canonical name을 확정하는 작업을 별도로 진행할 수 있다.


In [24]:
print("### 서비스_업종_코드 <-> 서비스_업종_코드_명 (sales, stores)")
for label, df in [("sales", sales), ("stores", stores)]:
    check_code_name_consistency(df, "서비스_업종_코드", "서비스_업종_코드_명", label)


### 서비스_업종_코드 <-> 서비스_업종_코드_명 (sales, stores)
[sales] 동일 서비스_업종_코드가 서로 다른 서비스_업종_코드_명과 연결된 사례: 0건
[sales] 동일 서비스_업종_코드_명이 서로 다른 서비스_업종_코드와 연결된 사례: 0건 (오류로 단정하지 않고 사례만 기록)


[stores] 동일 서비스_업종_코드가 서로 다른 서비스_업종_코드_명과 연결된 사례: 0건


[stores] 동일 서비스_업종_코드_명이 서로 다른 서비스_업종_코드와 연결된 사례: 0건 (오류로 단정하지 않고 사례만 기록)


## 11. 결측치 점검

핵심 key 컬럼에 결측이 있으면 데이터 연결(join) 자체가 불가능해지므로 별도로 검증한다. 그 외 컬럼에 결측이 있더라도, 그 의미(예: 실제 0인지 단순 미수집인지)를 확인할 근거가 없으면 임의로 0이나 평균으로 채우지 않고 `NaN` 상태를 그대로 유지한다.


In [25]:
def check_missing(df, label, key_cols):
    total_na = int(df.isna().sum().sum())
    print(f"\n[{label}] 전체 결측치 수: {total_na}")
    if total_na == 0:
        print(f"[{label}] 결측치 없음")
    else:
        na_counts = df.isna().sum()
        na_counts = na_counts[na_counts > 0]
        na_ratio = (na_counts / len(df) * 100).round(4)
        report = pd.DataFrame({"결측_개수": na_counts, "결측_비율(%)": na_ratio})
        display(report)

    key_na = df[key_cols].isna().sum()
    print(f"[{label}] 핵심 key 컬럼({key_cols}) 결측 수:")
    print(key_na.to_dict())
    assert int(key_na.sum()) == 0, f"{label}의 핵심 key 컬럼에 결측치가 존재합니다."


check_missing(sales, "sales", ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"])
check_missing(stores, "stores", ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"])
check_missing(population, "population", ["기준_년분기_코드", "상권_코드"])



[sales] 전체 결측치 수: 0
[sales] 결측치 없음
[sales] 핵심 key 컬럼(['기준_년분기_코드', '상권_코드', '서비스_업종_코드']) 결측 수:
{'기준_년분기_코드': 0, '상권_코드': 0, '서비스_업종_코드': 0}

[stores] 전체 결측치 수: 0
[stores] 결측치 없음
[stores] 핵심 key 컬럼(['기준_년분기_코드', '상권_코드', '서비스_업종_코드']) 결측 수:
{'기준_년분기_코드': 0, '상권_코드': 0, '서비스_업종_코드': 0}

[population] 전체 결측치 수: 0
[population] 결측치 없음
[population] 핵심 key 컬럼(['기준_년분기_코드', '상권_코드']) 결측 수:
{'기준_년분기_코드': 0, '상권_코드': 0}


**결측치 처리 방침**: 위 결과에서 결측치가 발견되지 않았다면(핵심 key 포함) 별도의 대체(imputation) 작업은 수행하지 않는다. 만약 향후 원본 데이터가 갱신되어 결측이 발생하더라도, key 컬럼 결측은 즉시 원인을 확인해야 하는 오류로 간주하고, 범주형 컬럼은 의미 확인 전 임의 대체를 금지하며, 수치형 컬럼은 "결측=0"이라는 근거가 명확한 경우에만 대체한다는 원칙을 유지한다.


## 12. 중복 데이터 점검

`01`에서는 연도/분기별 개별 파일 단위로 grain 중복이 없음을 확인했다. 파일을 통합하고 `stores_2025` 컬럼명을 정규화한 이후에도 동일한 grain key로 다시 중복을 확인해야, 통합 과정 자체에서 중복이 새로 생기지 않았는지 보장할 수 있다.


In [26]:
def check_grain_dup(df, key_cols, label):
    dup_count = int(df.duplicated(subset=key_cols).sum())
    print(f"[{label}] grain key {key_cols} 기준 중복: {dup_count}건")
    if dup_count > 0:
        display(df[df.duplicated(subset=key_cols, keep=False)].sort_values(key_cols).head(20))
    return dup_count


sales_dup = check_grain_dup(sales, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], "sales")
stores_dup = check_grain_dup(stores, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], "stores")
population_dup = check_grain_dup(population, ["기준_년분기_코드", "상권_코드"], "population")


[sales] grain key ['기준_년분기_코드', '상권_코드', '서비스_업종_코드'] 기준 중복: 0건


[stores] grain key ['기준_년분기_코드', '상권_코드', '서비스_업종_코드'] 기준 중복: 0건
[population] grain key ['기준_년분기_코드', '상권_코드'] 기준 중복: 0건


In [27]:
# grain key와 무관하게, 모든 컬럼 값이 완전히 동일한 행(exact duplicate)도 별도로 확인한다.
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    exact_dup = int(df.duplicated().sum())
    print(f"[{label}] 완전 동일 행(exact duplicate) 수: {exact_dup}")


[sales] 완전 동일 행(exact duplicate) 수: 0


[stores] 완전 동일 행(exact duplicate) 수: 0
[population] 완전 동일 행(exact duplicate) 수: 0


## 13. 값 범위 및 기본 품질 검증

본격적인 이상치 제거는 EDA 이후로 미루고, 여기서는 "0보다 작을 수 없는 값이 음수인가"처럼 **명백하게 불가능한 값**만 확인한다. 0은 정상값이므로 이상치로 취급하지 않으며, 매우 큰 값이 있다고 해서 제거하지 않는다.


In [28]:
def check_negative(df, cols, label):
    result = {}
    for c in cols:
        n_neg = int((df[c] < 0).sum())
        if n_neg > 0:
            result[c] = n_neg
    print(f"[{label}] 음수 값이 존재하는 컬럼: {result if result else '없음'}")
    return result


sales_amount_cols = [c for c in sales.columns if c.endswith("매출_금액")]
sales_count_cols = [c for c in sales.columns if c.endswith("매출_건수")]
print("sales 매출 금액 계열 컬럼 수:", len(sales_amount_cols), "/ 매출 건수 계열 컬럼 수:", len(sales_count_cols))

sales_neg_amount = check_negative(sales, sales_amount_cols, "sales 매출 금액 계열")
sales_neg_count = check_negative(sales, sales_count_cols, "sales 매출 건수 계열")


sales 매출 금액 계열 컬럼 수: 24 / 매출 건수 계열 컬럼 수: 24
[sales 매출 금액 계열] 음수 값이 존재하는 컬럼: 없음
[sales 매출 건수 계열] 음수 값이 존재하는 컬럼: 없음


In [29]:
stores_neg_cols = ["점포_수", "유사_업종_점포_수", "개업_점포_수", "폐업_점포_수", "프랜차이즈_점포_수"]
stores_neg_cols = [c for c in stores_neg_cols if c in stores.columns]
stores_neg = check_negative(stores, stores_neg_cols, "stores 점포 관련 컬럼")

for rate_col in ["개업_율", "폐업_률"]:
    n_neg = int((stores[rate_col] < 0).sum())
    n_over_100 = int((stores[rate_col] > 100).sum())
    print(f"[stores] {rate_col}: 음수 {n_neg}건, 100 초과 {n_over_100}건 "
          f"(0은 정상값이므로 이상치로 취급하지 않음. 100 초과 값은 삭제하지 않고 '확인 필요' 사항으로만 기록)")


[stores 점포 관련 컬럼] 음수 값이 존재하는 컬럼: 없음
[stores] 개업_율: 음수 0건, 100 초과 14건 (0은 정상값이므로 이상치로 취급하지 않음. 100 초과 값은 삭제하지 않고 '확인 필요' 사항으로만 기록)
[stores] 폐업_률: 음수 0건, 100 초과 723건 (0은 정상값이므로 이상치로 취급하지 않음. 100 초과 값은 삭제하지 않고 '확인 필요' 사항으로만 기록)


In [30]:
population_value_cols = [c for c in population.columns if c.endswith("유동인구_수")]
print("population 유동인구 계열 컬럼 수:", len(population_value_cols))

population_neg = check_negative(population, population_value_cols, "population 유동인구 계열")


population 유동인구 계열 컬럼 수: 22
[population 유동인구 계열] 음수 값이 존재하는 컬럼: 없음


## 14. 구성 합계 일관성 검사

데이터가 손상되지 않았는지 확인하기 위해, 성별/연령대별/시간대별로 나뉜 값의 합이 전체 합계(`당월_매출_금액`, `당월_매출_건수`, `총_유동인구_수`)와 대략 일치하는지 점검한다. 공공데이터는 반올림이나 별도 추정 로직 때문에 완전히 일치하지 않을 수 있으므로, 불일치가 있다고 해서 값을 임의로 수정하지 않고 품질 참고 지표로만 요약한다.


In [31]:
def summarize_diff(actual, computed_sum, label):
    diff = actual - computed_sum
    exact_ratio = float((diff == 0).mean() * 100)
    print(f"[{label}]")
    print(f"  완전 일치 비율     : {exact_ratio:.2f}%")
    print(f"  차이의 중앙값       : {diff.median()}")
    print(f"  절대 차이의 중앙값  : {diff.abs().median()}")
    print(f"  최대 절대 차이      : {diff.abs().max()}")
    return diff


gender_amount_cols = [c for c in ["남성_매출_금액", "여성_매출_금액"] if c in sales.columns]
age_amount_cols = [c for c in sales.columns if c.startswith("연령대_") and c.endswith("_매출_금액")]
time_amount_cols = [c for c in sales.columns if c.startswith("시간대_") and c.endswith("_매출_금액")]
gender_count_cols = [c for c in ["남성_매출_건수", "여성_매출_건수"] if c in sales.columns]
age_count_cols = [c for c in sales.columns if c.startswith("연령대_") and c.endswith("_매출_건수")]
time_count_cols = [c for c in sales.columns if c.startswith("시간대_") and c.endswith("_매출_건수")]

print("성별 매출금액 컬럼:", gender_amount_cols)
print("연령대별 매출금액 컬럼:", age_amount_cols)
print("시간대별 매출금액 컬럼:", time_amount_cols)
print("성별 매출건수 컬럼:", gender_count_cols)
print("연령대별 매출건수 컬럼:", age_count_cols)
print("시간대별 매출건수 컬럼:", time_count_cols)


성별 매출금액 컬럼: ['남성_매출_금액', '여성_매출_금액']
연령대별 매출금액 컬럼: ['연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액', '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액']
시간대별 매출금액 컬럼: ['시간대_00~06_매출_금액', '시간대_06~11_매출_금액', '시간대_11~14_매출_금액', '시간대_14~17_매출_금액', '시간대_17~21_매출_금액', '시간대_21~24_매출_금액']
성별 매출건수 컬럼: ['남성_매출_건수', '여성_매출_건수']
연령대별 매출건수 컬럼: ['연령대_10_매출_건수', '연령대_20_매출_건수', '연령대_30_매출_건수', '연령대_40_매출_건수', '연령대_50_매출_건수', '연령대_60_이상_매출_건수']
시간대별 매출건수 컬럼: ['시간대_00~06_매출_건수', '시간대_06~11_매출_건수', '시간대_11~14_매출_건수', '시간대_14~17_매출_건수', '시간대_17~21_매출_건수', '시간대_21~24_매출_건수']


In [32]:
if len(gender_amount_cols) == 2:
    _ = summarize_diff(sales["당월_매출_금액"], sales[gender_amount_cols].sum(axis=1),
                        "sales: 남성+여성 매출금액 vs 당월_매출_금액")

if age_amount_cols:
    _ = summarize_diff(sales["당월_매출_금액"], sales[age_amount_cols].sum(axis=1),
                        "sales: 연령대별 매출금액 합 vs 당월_매출_금액")

if time_amount_cols:
    _ = summarize_diff(sales["당월_매출_금액"], sales[time_amount_cols].sum(axis=1),
                        "sales: 시간대별 매출금액 합 vs 당월_매출_금액")


[sales: 남성+여성 매출금액 vs 당월_매출_금액]
  완전 일치 비율     : 41.03%
  차이의 중앙값       : 1492888.0
  절대 차이의 중앙값  : 1492888.0
  최대 절대 차이      : 464174265427
[sales: 연령대별 매출금액 합 vs 당월_매출_금액]
  완전 일치 비율     : 40.95%
  차이의 중앙값       : 1492714.0
  절대 차이의 중앙값  : 1492714.0
  최대 절대 차이      : 464174265431
[sales: 시간대별 매출금액 합 vs 당월_매출_금액]
  완전 일치 비율     : 100.00%
  차이의 중앙값       : 0.0
  절대 차이의 중앙값  : 0.0
  최대 절대 차이      : 0


In [33]:
if len(gender_count_cols) == 2:
    _ = summarize_diff(sales["당월_매출_건수"], sales[gender_count_cols].sum(axis=1),
                        "sales: 남성+여성 매출건수 vs 당월_매출_건수")

if age_count_cols:
    _ = summarize_diff(sales["당월_매출_건수"], sales[age_count_cols].sum(axis=1),
                        "sales: 연령대별 매출건수 합 vs 당월_매출_건수")

if time_count_cols:
    _ = summarize_diff(sales["당월_매출_건수"], sales[time_count_cols].sum(axis=1),
                        "sales: 시간대별 매출건수 합 vs 당월_매출_건수")


[sales: 남성+여성 매출건수 vs 당월_매출_건수]
  완전 일치 비율     : 40.79%


  차이의 중앙값       : 23.0
  절대 차이의 중앙값  : 23.0
  최대 절대 차이      : 1334219


[sales: 연령대별 매출건수 합 vs 당월_매출_건수]
  완전 일치 비율     : 40.58%
  차이의 중앙값       : 23.0


  절대 차이의 중앙값  : 23.0
  최대 절대 차이      : 1334216
[sales: 시간대별 매출건수 합 vs 당월_매출_건수]
  완전 일치 비율     : 100.00%
  차이의 중앙값       : 0.0


  절대 차이의 중앙값  : 0.0
  최대 절대 차이      : 1


In [34]:
gender_pop_cols = [c for c in ["남성_유동인구_수", "여성_유동인구_수"] if c in population.columns]
age_pop_cols = [c for c in population.columns if c.startswith("연령대_") and c.endswith("_유동인구_수")]
time_pop_cols = [c for c in population.columns if c.startswith("시간대_") and c.endswith("_유동인구_수")]

print("성별 유동인구 컬럼:", gender_pop_cols)
print("연령대별 유동인구 컬럼:", age_pop_cols)
print("시간대별 유동인구 컬럼:", time_pop_cols)

if len(gender_pop_cols) == 2:
    _ = summarize_diff(population["총_유동인구_수"], population[gender_pop_cols].sum(axis=1),
                        "population: 남성+여성 유동인구 vs 총_유동인구_수")

if age_pop_cols:
    _ = summarize_diff(population["총_유동인구_수"], population[age_pop_cols].sum(axis=1),
                        "population: 연령대별 유동인구 합 vs 총_유동인구_수")

if time_pop_cols:
    _ = summarize_diff(population["총_유동인구_수"], population[time_pop_cols].sum(axis=1),
                        "population: 시간대별 유동인구 합 vs 총_유동인구_수")


성별 유동인구 컬럼: ['남성_유동인구_수', '여성_유동인구_수']
연령대별 유동인구 컬럼: ['연령대_10_유동인구_수', '연령대_20_유동인구_수', '연령대_30_유동인구_수', '연령대_40_유동인구_수', '연령대_50_유동인구_수', '연령대_60_이상_유동인구_수']
시간대별 유동인구 컬럼: ['시간대_00_06_유동인구_수', '시간대_06_11_유동인구_수', '시간대_11_14_유동인구_수', '시간대_14_17_유동인구_수', '시간대_17_21_유동인구_수', '시간대_21_24_유동인구_수']
[population: 남성+여성 유동인구 vs 총_유동인구_수]
  완전 일치 비율     : 49.59%
  차이의 중앙값       : 0.0
  절대 차이의 중앙값  : 1.0
  최대 절대 차이      : 3
[population: 연령대별 유동인구 합 vs 총_유동인구_수]
  완전 일치 비율     : 30.47%
  차이의 중앙값       : 0.0
  절대 차이의 중앙값  : 1.0
  최대 절대 차이      : 5
[population: 시간대별 유동인구 합 vs 총_유동인구_수]
  완전 일치 비율     : 30.83%
  차이의 중앙값       : 0.0
  절대 차이의 중앙값  : 1.0
  최대 절대 차이      : 5


성별·연령대별 세부 합계와 당월/총 합계 사이에는 위와 같은 차이가 존재한다. sales의 시간대별 합계는 당월 합계와 매우 높은 수준으로 일치하는 반면, 성별·연령대별 합계는 상당수 행에서 정확히 일치하지 않았다. 이 차이는 세부 항목의 집계 기준, 데이터 산출 방식, 추정·비식별화 처리 등 여러 원인 중 하나이거나 복합적으로 작용한 결과일 수 있으나, 현재 데이터만으로는 정확한 원인을 확정할 수 없다.

따라서 이 단계에서는 값을 임의로 맞추거나 상권 특성으로 해석하지 않고, 차이의 크기(완전 일치 비율·중앙값·최대값)만 사실로 기록해둔다. 향후 예를 들어 `(연령대_20_매출_금액 + 연령대_30_매출_금액) / 당월_매출_금액`과 같은 비율형 파생지표를 만들기 전에는, 서울시 상권분석서비스의 성별·연령대별 매출 컬럼과 당월 매출 컬럼이 동일한 산출 기준으로 집계되는지 공식 컬럼 정의를 통해 추가로 확인할 필요가 있다.


## 15. 기간 및 서울 전체 상권 커버리지 검증

전처리(컬럼 정규화, 타입 변환, rename) 과정에서 특정 분기나 대규모 상권 데이터가 실수로 손실되지 않았는지 다시 확인한다. 이 결과는 순수하게 품질 검증 목적이며, 상권 수의 시계열 변화 자체를 해석하지 않는다 (그 해석은 `03_eda.ipynb`에서 다룬다).


In [35]:
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    quarters_present = sorted(df["기준_년분기_코드"].unique().tolist())
    missing = sorted(set(expected_quarters) - set(quarters_present))
    ok = (len(quarters_present) == 20) and (not missing)
    print(f"[{label}] 20개 분기 모두 존재: {ok} (누락 분기: {missing if missing else '없음'})")
    assert ok, f"{label}에서 20개 분기가 모두 확인되지 않았습니다."


[sales] 20개 분기 모두 존재: True (누락 분기: 없음)
[stores] 20개 분기 모두 존재: True (누락 분기: 없음)
[population] 20개 분기 모두 존재: True (누락 분기: 없음)


In [36]:
print("분기별 고유 상권 수 (품질 검증용 참고 자료, 해석하지 않음)")
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    per_quarter = df.groupby("기준_년분기_코드")["상권_코드"].nunique()
    print(f"\n[{label}]")
    print(per_quarter)


분기별 고유 상권 수 (품질 검증용 참고 자료, 해석하지 않음)

[sales]
기준_년분기_코드
20211    1574
20212    1573
20213    1571
20214    1570
20221    1570
20222    1571
20223    1568
20224    1570
20231    1573
20232    1574
20233    1574
20234    1572
20241    1570
20242    1571
20243    1568
20244    1570
20251    1572
20252    1570
20253    1569
20254    1565
Name: 상권_코드, dtype: int64

[stores]
기준_년분기_코드
20211    1650
20212    1650
20213    1650
20214    1650
20221    1650
20222    1650
20223    1650
20224    1650
20231    1650
20232    1650
20233    1650
20234    1650
20241    1650
20242    1650
20243    1650
20244    1650
20251    1650
20252    1650
20253    1650
20254    1650
Name: 상권_코드, dtype: int64

[population]
기준_년분기_코드
20211    1650
20212    1650
20213    1650
20214    1650
20221    1650
20222    1650
20223    1650
20224    1649
20231    1649
20232    1649
20233    1649
20234    1648
20241    1649
20242    1649
20243    1648
20244    1649
20251    1650
20252    1649
20253    1648
20254    1648
Name: 상권_

In [37]:
print("전체 기간(2021Q1~2025Q4) 동안 등장한 고유 상권코드 수")
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    print(f"[{label}] 고유 상권코드 수: {df['상권_코드'].nunique()}")


전체 기간(2021Q1~2025Q4) 동안 등장한 고유 상권코드 수
[sales] 고유 상권코드 수: 1596
[stores] 고유 상권코드 수: 1650
[population] 고유 상권코드 수: 1650


## 16. 황학 관련 데이터 최종 검증

`01`에서 확인한 황학 관련 4개 상권코드(3110055, 3110057, 3130054, 3130055)가 전처리 이후에도 세 데이터 모두에서 20개 분기 전체에 걸쳐 유지되는지 재확인한다. **이 검증은 황학동으로 데이터를 필터링하기 위한 것이 아니라, 전처리 과정에서 핵심 분석 대상 데이터가 손실되지 않았는지 확인하기 위한 것**이며, 검증 이후에도 최종 저장 데이터는 서울 전체 상권을 그대로 유지한다.


In [38]:
HWANGHAK_CODES = [3110055, 3110057, 3130054, 3130055]

hwanghak_rows = []
for label, df in [("sales", sales), ("stores", stores), ("population", population)]:
    for code in HWANGHAK_CODES:
        sub = df[df["상권_코드"] == code]
        exists = len(sub) > 0
        n_quarters = int(sub["기준_년분기_코드"].nunique())
        complete = (n_quarters == 20)
        hwanghak_rows.append({
            "상권코드": code,
            "데이터": label,
            "존재 여부": exists,
            "고유 분기 수": n_quarters,
            "20개 분기 완전성": complete,
        })

hwanghak_check = pd.DataFrame(hwanghak_rows)
display(hwanghak_check)

assert bool(hwanghak_check["존재 여부"].all()), "황학 4개 상권 중 일부가 누락되었습니다."
assert bool(hwanghak_check["20개 분기 완전성"].all()), "황학 4개 상권 중 일부가 20개 분기를 모두 채우지 못했습니다."
print("\n황학 4개 상권 모두 sales/stores/population에서 20개 분기 전체 유지 확인 완료.")
print("(참고: 이 검증은 필터링을 위한 것이 아니며, 최종 저장 데이터는 서울 전체 상권을 그대로 포함한다.)")


,상권코드,데이터,존재 여부,고유 분기 수,20개 분기 완전성
0,3110055,sales,True,20,True
1,3110057,sales,True,20,True
2,3130054,sales,True,20,True
3,3130055,sales,True,20,True
4,3110055,stores,True,20,True
5,3110057,stores,True,20,True
6,3130054,stores,True,20,True
7,3130055,stores,True,20,True
8,3110055,population,True,20,True
9,3110057,population,True,20,True



황학 4개 상권 모두 sales/stores/population에서 20개 분기 전체 유지 확인 완료.
(참고: 이 검증은 필터링을 위한 것이 아니며, 최종 저장 데이터는 서울 전체 상권을 그대로 포함한다.)


## 17. 데이터 간 key 정합성 재검증

전처리 이후에도 `sales ↔ stores`(기준년분기+상권코드+서비스업종), `sales/stores ↔ population`(기준년분기+상권코드) 사이의 key 매칭률이 `01`에서 확인한 결과와 논리적으로 일관되는지 재확인한다. 이는 이후 Tableau에서 여러 데이터셋을 관계(Relationship)로 연결하거나 AI가 상권별 매출·점포·인구 정보를 함께 조회할 때 잘못된 연결이 발생하지 않도록 하기 위한 핵심 검증이다. 특히 `sales -> stores` 매칭률은 `01`에서 100%였으므로, 전처리 후에도 100%가 아니라면 저장하지 않고 원인을 먼저 확인해야 한다.


In [39]:
def key_match(df_a, cols_a, df_b, cols_b, name_a, name_b):
    keys_a = set(map(tuple, df_a[cols_a].drop_duplicates().itertuples(index=False, name=None)))
    keys_b = set(map(tuple, df_b[cols_b].drop_duplicates().itertuples(index=False, name=None)))
    common = keys_a & keys_b
    rate_a = (len(common) / len(keys_a) * 100) if keys_a else None
    rate_b = (len(common) / len(keys_b) * 100) if keys_b else None
    print(f"{name_a} unique key 수 : {len(keys_a)}")
    print(f"{name_b} unique key 수 : {len(keys_b)}")
    print(f"공통 key 수           : {len(common)}")
    print(f"{name_a} 기준 매칭률   : {rate_a:.2f}%" if rate_a is not None else f"{name_a} 기준 매칭률: N/A")
    print(f"{name_b} 기준 매칭률   : {rate_b:.2f}%" if rate_b is not None else f"{name_b} 기준 매칭률: N/A")
    return rate_a, rate_b


print("### sales -> stores / stores -> sales (기준년분기 + 상권코드 + 서비스업종)")
sales_to_stores_rate, stores_to_sales_rate = key_match(
    sales, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
    stores, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
    "sales", "stores",
)
assert sales_to_stores_rate is not None and abs(sales_to_stores_rate - 100.0) < 1e-9, (
    "sales -> stores key 매칭률이 100%가 아닙니다. 저장하지 말고 원인을 먼저 확인해야 합니다."
)


### sales -> stores / stores -> sales (기준년분기 + 상권코드 + 서비스업종)


sales unique key 수 : 439141
stores unique key 수 : 1528872
공통 key 수           : 439141
sales 기준 매칭률   : 100.00%
stores 기준 매칭률   : 28.72%


In [40]:
print("### sales -> population (기준년분기 + 상권코드)")
sales_to_population_rate, population_to_sales_rate = key_match(
    sales, ["기준_년분기_코드", "상권_코드"],
    population, ["기준_년분기_코드", "상권_코드"],
    "sales", "population",
)


### sales -> population (기준년분기 + 상권코드)
sales unique key 수 : 31415
population unique key 수 : 32984
공통 key 수           : 31399
sales 기준 매칭률   : 99.95%
population 기준 매칭률   : 95.19%


In [41]:
print("### stores -> population (기준년분기 + 상권코드)")
stores_to_population_rate, population_to_stores_rate = key_match(
    stores, ["기준_년분기_코드", "상권_코드"],
    population, ["기준_년분기_코드", "상권_코드"],
    "stores", "population",
)


### stores -> population (기준년분기 + 상권코드)
stores unique key 수 : 33000
population unique key 수 : 32984
공통 key 수           : 32984
stores 기준 매칭률   : 99.95%
population 기준 매칭률   : 100.00%


### 17-1. population과 매칭되지 않는 `(기준_년분기_코드, 상권_코드)` key 상세 확인

위 매칭률만으로는 정확히 어떤 `(기준_년분기_코드, 상권_코드)` 조합이 서로 어긋나는지 알 수 없다. 이후 Tableau relationship이나 AI 질의에서 특정 분기·상권 조합이 조용히 누락되는 것을 막기 위해, 매칭되지 않는 실제 key를 sales/stores/population 세 방향 모두에서 명시적으로 확인한다. 상권명은 참고용으로만 붙이며(상권코드 기준으로 첫 번째로 등장한 이름 하나만 매핑하여 grain이 늘어나거나 중복이 생기지 않도록 한다), 이 단계에서는 어떤 key가 일치하지 않는지 사실만 기록할 뿐 행을 삭제하거나 population 값을 임의로 보완하지 않는다.


In [42]:
POP_JOIN_KEY = ["기준_년분기_코드", "상권_코드"]

sales_qc_df = sales[POP_JOIN_KEY].drop_duplicates()
stores_qc_df = stores[POP_JOIN_KEY].drop_duplicates()
population_qc_df = population[POP_JOIN_KEY].drop_duplicates()


def area_name_lookup(df):
    """상권_코드 -> 상권_코드_명 조회용 Series. 코드당 첫 번째로 등장한 이름 하나만 사용해
    표시용 참고 정보만 붙이고 grain(행 수)을 늘리지 않는다."""
    return df.drop_duplicates(subset="상권_코드").set_index("상권_코드")["상권_코드_명"]


# 1-1. sales에는 있지만 population에는 없는 key
sales_only_population_missing = sales_qc_df.merge(
    population_qc_df, on=POP_JOIN_KEY, how="left", indicator=True
)
sales_only_population_missing = sales_only_population_missing.loc[
    sales_only_population_missing["_merge"] == "left_only", POP_JOIN_KEY
].copy()
sales_only_population_missing["상권_코드_명"] = sales_only_population_missing["상권_코드"].map(area_name_lookup(sales))

print(f"[sales -> population] 미매칭 key 개수: {len(sales_only_population_missing)}")
display(sales_only_population_missing.sort_values(POP_JOIN_KEY))


[sales -> population] 미매칭 key 개수: 16


,기준_년분기_코드,상권_코드,상권_코드_명
11896,20224,3110946,청계산원터골
13469,20231,3110948,헌인가구단지
15041,20232,3110946,청계산원터골
16614,20233,3110946,청계산원터골
18187,20234,3110946,청계산원터골
18189,20234,3110948,헌인가구단지
19755,20241,3110946,청계산원터골
21327,20242,3110946,청계산원터골
22896,20243,3110946,청계산원터골
22948,20243,3111002,한국교통안전공단 강남자동차검사소


In [43]:
# 1-2. stores에는 있지만 population에는 없는 key
stores_only_population_missing = stores_qc_df.merge(
    population_qc_df, on=POP_JOIN_KEY, how="left", indicator=True
)
stores_only_population_missing = stores_only_population_missing.loc[
    stores_only_population_missing["_merge"] == "left_only", POP_JOIN_KEY
].copy()
stores_only_population_missing["상권_코드_명"] = stores_only_population_missing["상권_코드"].map(area_name_lookup(stores))

print(f"[stores -> population] 미매칭 key 개수: {len(stores_only_population_missing)}")
display(stores_only_population_missing.sort_values(POP_JOIN_KEY))


[stores -> population] 미매칭 key 개수: 16


,기준_년분기_코드,상권_코드,상권_코드_명
12495,20224,3110946,청계산원터골
14147,20231,3110948,헌인가구단지
15795,20232,3110946,청계산원터골
17445,20233,3110946,청계산원터골
19095,20234,3110946,청계산원터골
19097,20234,3110948,헌인가구단지
20745,20241,3110946,청계산원터골
22395,20242,3110946,청계산원터골
24045,20243,3110946,청계산원터골
24101,20243,3111002,한국교통안전공단 강남자동차검사소


In [44]:
# 1-3. population에는 있지만 sales에는 없는 key
# 이 방향은 오류로 판단하지 않고, 데이터 수록 범위 차이를 확인하는 품질 참고 자료로만 사용한다.
population_only_sales_missing = population_qc_df.merge(
    sales_qc_df, on=POP_JOIN_KEY, how="left", indicator=True
)
population_only_sales_missing = population_only_sales_missing.loc[
    population_only_sales_missing["_merge"] == "left_only", POP_JOIN_KEY
].copy()
population_only_sales_missing["상권_코드_명"] = population_only_sales_missing["상권_코드"].map(area_name_lookup(population))

print(f"[population -> sales] sales에 없는 population key 개수: {len(population_only_sales_missing)} "
      f"(오류가 아니라 데이터 수록 범위 차이 확인용 참고 자료)")
display(population_only_sales_missing.sort_values(POP_JOIN_KEY).head(30))

print(f"\n[요약] sales에만 있는 key: {len(sales_only_population_missing)}건 / "
      f"stores에만 있는 key: {len(stores_only_population_missing)}건 / "
      f"population에만 있는 key(sales 기준): {len(population_only_sales_missing)}건")


[population -> sales] sales에 없는 population key 개수: 1585 (오류가 아니라 데이터 수록 범위 차이 확인용 참고 자료)


,기준_년분기_코드,상권_코드,상권_코드_명
1083,20211,3110011,청운초등학교
1119,20211,3110047,다산성곽길
1141,20211,3110070,용산세무서
1159,20211,3110086,이태원역 북측
755,20211,3110097,한남초등학교
1171,20211,3110100,옥수동우편취급국
1172,20211,3110101,옥정중학교
1173,20211,3110108,성동구립금호도서관
1182,20211,3110111,대현산장미원
1190,20211,3110119,마장지하차도



[요약] sales에만 있는 key: 16건 / stores에만 있는 key: 16건 / population에만 있는 key(sales 기준): 1585건


`01`에서도 sales/stores 기준 population 매칭률이 100%가 아니었다(sales 기준 약 99.95%, population 기준 약 95.19%). 이번 통합 이후 실제 unique key 수를 비교하면 `sales(31,415) < population(32,984) < stores(33,000)` 순으로, 세 데이터의 `(기준_년분기_코드, 상권_코드)` 수록 범위가 서로 완전히 같지 않다.

- **sales -> stores**: 매칭률 100.00%로 `01`과 동일하게 완전히 유지된다.
- **sales -> population**: sales에만 있고 population에는 없는 key가 16건 있다(상권코드 3110946 청계산원터골, 3110948 헌인가구단지, 3111002 한국교통안전공단 강남자동차검사소의 일부 분기). 반대로 population에는 있지만 sales에는 없는 key는 1,585건으로(청운초등학교, 다산성곽길 등 소규모/비상업 성격의 지점이 다수 포함), sales/stores에는 애초에 매출·점포 실적이 보고되지 않는 상권으로 추정된다.
- **stores -> population**: stores에만 있고 population에는 없는 key도 동일하게 16건이며(위와 같은 3개 상권코드), population은 stores 매칭률 기준 100.00%로 stores의 (분기, 상권) key 집합 안에 완전히 포함된다.

즉 population 쪽이 sales/stores보다 일방적으로 더 많은 것이 아니라 **sales < population < stores** 관계이며, population에만 존재하는 다수의 key(1,585건)와 sales/stores에만 존재하는 소수의 key(각 16건, 동일한 3개 상권)가 함께 존재한다. 이는 population의 조사 대상 상권 범위가 sales/stores와 다르게 정의되어 있을 가능성을 보여주는 품질 참고 정보이며, 이번 단계에서는 미매칭 행을 삭제하거나 population 값을 임의로 채우지 않는다.


## 18. 분석 준비도 검증

저장하기 전에, 이후 분석(EDA, Tableau, 유사 상권 비교, AI 진단)에 필요한 최소 조건을 boolean으로 정리하여 최종 확인한다. 하나라도 실패하면 저장하지 않고 무엇이 실패했는지 명확히 출력한다. 반면 구성 합계 차이나 폐업률 100% 초과처럼 공공데이터 자체 특성으로 발생할 수 있는 사항은 저장을 막지 않는 별도의 경고로 구분한다.


In [45]:
readiness = {
    "20개 분기 모두 존재 (sales)": sorted(sales["기준_년분기_코드"].unique().tolist()) == expected_quarters,
    "20개 분기 모두 존재 (stores)": sorted(stores["기준_년분기_코드"].unique().tolist()) == expected_quarters,
    "20개 분기 모두 존재 (population)": sorted(population["기준_년분기_코드"].unique().tolist()) == expected_quarters,
    "sales grain key 중복 없음": sales_dup == 0,
    "stores grain key 중복 없음": stores_dup == 0,
    "population grain key 중복 없음": population_dup == 0,
    "sales -> stores key 매칭률 100%": sales_to_stores_rate is not None and abs(sales_to_stores_rate - 100.0) < 1e-9,
    "황학 4개 상권 모두 존재": bool(hwanghak_check["존재 여부"].all()),
    "황학 4개 상권 각각 20개 분기 존재": bool(hwanghak_check["20개 분기 완전성"].all()),
    "상권_코드 dtype 3개 데이터 일치": sales["상권_코드"].dtype == stores["상권_코드"].dtype == population["상권_코드"].dtype,
    "서비스_업종_코드 dtype sales/stores 일치": sales["서비스_업종_코드"].dtype == stores["서비스_업종_코드"].dtype,
    "핵심 key 컬럼 결측 없음": bool(
        (sales[["기준_년분기_코드", "상권_코드", "서비스_업종_코드"]].isna().sum().sum() == 0)
        and (stores[["기준_년분기_코드", "상권_코드", "서비스_업종_코드"]].isna().sum().sum() == 0)
        and (population[["기준_년분기_코드", "상권_코드"]].isna().sum().sum() == 0)
    ),
}

readiness_df = pd.Series(readiness, name="통과 여부").to_frame()
display(readiness_df)

all_pass = bool(readiness_df["통과 여부"].all())
print("\n필수 검증 전체 통과 여부:", all_pass)

if not all_pass:
    failed = readiness_df[~readiness_df["통과 여부"]]
    print("\n실패한 항목:")
    display(failed)
    raise AssertionError("필수 분석 준비도 검증에 실패한 항목이 있어 저장을 진행하지 않습니다. 실패 항목을 먼저 확인하세요.")

print("\n[참고: 저장을 막지 않는 경고사항 - 공공데이터 특성상 발생 가능]")
print("- 성별/연령대별/시간대별 구성 합계가 당월/총 합계와 완전히 일치하지 않는 행이 존재할 수 있음 (섹션 14 참고)")
print("- stores의 개업_율/폐업_률 중 100을 초과하는 값이 존재할 수 있음 (섹션 13 참고, 삭제하지 않고 그대로 유지)")
print("- sales/stores 기준 population과의 key 매칭률이 100%가 아닐 수 있음 (섹션 17 참고)")


,통과 여부
20개 분기 모두 존재 (sales),True
20개 분기 모두 존재 (stores),True
20개 분기 모두 존재 (population),True
sales grain key 중복 없음,True
stores grain key 중복 없음,True
population grain key 중복 없음,True
sales -> stores key 매칭률 100%,True
황학 4개 상권 모두 존재,True
황학 4개 상권 각각 20개 분기 존재,True
상권_코드 dtype 3개 데이터 일치,True



필수 검증 전체 통과 여부: True

[참고: 저장을 막지 않는 경고사항 - 공공데이터 특성상 발생 가능]
- 성별/연령대별/시간대별 구성 합계가 당월/총 합계와 완전히 일치하지 않는 행이 존재할 수 있음 (섹션 14 참고)
- stores의 개업_율/폐업_률 중 100을 초과하는 값이 존재할 수 있음 (섹션 13 참고, 삭제하지 않고 그대로 유지)
- sales/stores 기준 population과의 key 매칭률이 100%가 아닐 수 있음 (섹션 17 참고)


## 19. processed 데이터 저장

필수 검증을 모두 통과했으므로, grain이 서로 다른 세 데이터를 하나로 강제 병합하지 않고 각각 별도 CSV로 `data/processed/`에 저장한다. 한글 컬럼명과 Tableau 호환성을 고려해 `encoding="utf-8-sig"`, `index=False`를 사용한다.


In [46]:
sales.to_csv(PROCESSED_DIR / "sales.csv", encoding="utf-8-sig", index=False)
stores.to_csv(PROCESSED_DIR / "stores.csv", encoding="utf-8-sig", index=False)
population.to_csv(PROCESSED_DIR / "population.csv", encoding="utf-8-sig", index=False)

print("저장 완료:", sorted(p.name for p in PROCESSED_DIR.glob("*.csv")))


저장 완료: ['population.csv', 'sales.csv', 'stores.csv']


In [47]:
def verify_saved(path, original_df, key_cols, label):
    """저장 전 DataFrame과 저장 후 재로드한 DataFrame이 shape, 컬럼명/순서, 핵심 key dtype,
    key(개별 컬럼 및 grain 조합) unique 개수까지 모두 동일한지 명시적으로 assert 검증한다."""
    reloaded = pd.read_csv(path, encoding="utf-8-sig")
    print(f"\n[{label}] 원본 shape: {original_df.shape} / 재로드 shape: {reloaded.shape}")

    # 4-1. shape
    assert original_df.shape == reloaded.shape, f"{label}: 저장 전후 shape가 다릅니다."

    # 4-2. 컬럼명 및 순서
    assert original_df.columns.tolist() == reloaded.columns.tolist(), \
        f"{label}: 저장 전후 컬럼명/순서가 다릅니다."

    # 4-3. 핵심 key dtype
    for col in key_cols:
        orig_dtype = original_df[col].dtype
        reload_dtype = reloaded[col].dtype
        print(f"  - {col}: dtype 원본={orig_dtype}, dtype 재로드={reload_dtype}")
        assert orig_dtype == reload_dtype, \
            f"{label}.{col}: 저장 전후 dtype이 다릅니다 ({orig_dtype} -> {reload_dtype})."

    # 4-4. 컬럼별 unique 개수 + grain key 조합 unique 개수
    for col in key_cols:
        orig_n = original_df[col].nunique()
        reload_n = reloaded[col].nunique()
        print(f"  - {col}: 원본 unique={orig_n}, 재로드 unique={reload_n}")
        assert orig_n == reload_n, f"{label}.{col}: 저장 전후 unique 개수가 다릅니다."

    orig_grain_n = len(original_df[key_cols].drop_duplicates())
    reload_grain_n = len(reloaded[key_cols].drop_duplicates())
    print(f"  - grain key {key_cols} 조합 unique 개수: 원본={orig_grain_n}, 재로드={reload_grain_n}")
    assert orig_grain_n == reload_grain_n, \
        f"{label}: 저장 전후 grain key 조합 unique 개수가 다릅니다."

    del reloaded
    print(f"[{label}] 저장 후 무결성 검증 통과")


verify_saved(PROCESSED_DIR / "sales.csv", sales, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], "sales")
verify_saved(PROCESSED_DIR / "stores.csv", stores, ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"], "stores")
verify_saved(PROCESSED_DIR / "population.csv", population, ["기준_년분기_코드", "상권_코드"], "population")

print("\n저장 후 무결성 검증 통과 (shape / 컬럼명·순서 / key dtype / grain key unique 개수 모두 일치)")



[sales] 원본 shape: (439141, 55) / 재로드 shape: (439141, 55)
  - 기준_년분기_코드: dtype 원본=int64, dtype 재로드=int64
  - 상권_코드: dtype 원본=int64, dtype 재로드=int64
  - 서비스_업종_코드: dtype 원본=str, dtype 재로드=str
  - 기준_년분기_코드: 원본 unique=20, 재로드 unique=20
  - 상권_코드: 원본 unique=1596, 재로드 unique=1596
  - 서비스_업종_코드: 원본 unique=63, 재로드 unique=63
  - grain key ['기준_년분기_코드', '상권_코드', '서비스_업종_코드'] 조합 unique 개수: 원본=439141, 재로드=439141
[sales] 저장 후 무결성 검증 통과



[stores] 원본 shape: (1528872, 14) / 재로드 shape: (1528872, 14)
  - 기준_년분기_코드: dtype 원본=int64, dtype 재로드=int64
  - 상권_코드: dtype 원본=int64, dtype 재로드=int64
  - 서비스_업종_코드: dtype 원본=str, dtype 재로드=str
  - 기준_년분기_코드: 원본 unique=20, 재로드 unique=20
  - 상권_코드: 원본 unique=1650, 재로드 unique=1650
  - 서비스_업종_코드: 원본 unique=100, 재로드 unique=100


  - grain key ['기준_년분기_코드', '상권_코드', '서비스_업종_코드'] 조합 unique 개수: 원본=1528872, 재로드=1528872
[stores] 저장 후 무결성 검증 통과

[population] 원본 shape: (32984, 27) / 재로드 shape: (32984, 27)
  - 기준_년분기_코드: dtype 원본=int64, dtype 재로드=int64
  - 상권_코드: dtype 원본=int64, dtype 재로드=int64
  - 기준_년분기_코드: 원본 unique=20, 재로드 unique=20
  - 상권_코드: 원본 unique=1650, 재로드 unique=1650
  - grain key ['기준_년분기_코드', '상권_코드'] 조합 unique 개수: 원본=32984, 재로드=32984
[population] 저장 후 무결성 검증 통과

저장 후 무결성 검증 통과 (shape / 컬럼명·순서 / key dtype / grain key unique 개수 모두 일치)


## 20. 최종 전처리 요약

| 데이터 | 원본 파일 수 | 통합 후 행 수 | 컬럼 수 | 기간 | 상권 수 | 결측치 | key 중복 | 저장 파일 |
|---|---:|---:|---:|---|---:|---|---|---|
| sales | 5 (연도별) | 439,141 | 55 | 2021Q1~2025Q4 | 1,596 | 없음 | 없음 | `data/processed/sales.csv` |
| stores | 5 (연도별) | 1,528,872 | 14 | 2021Q1~2025Q4 | 1,650 | 없음 | 없음 | `data/processed/stores.csv` |
| population | 20 (분기별) | 32,984 | 27 | 2021Q1~2025Q4 | 1,650 | 없음 | 없음 | `data/processed/population.csv` |

### 실행 결과 기반 정리

1. **stores_2025 컬럼명 차이 처리**: `STORES_2025_COLUMN_MAP` 명시적 dictionary로 `stdr_yyqu_cd` 등 14개 영문 컬럼명을 2021~2024와 동일한 한글 컬럼명으로 rename했다. rename 전 매핑 dict의 키와 실제 `stores_2025.csv` 컬럼 집합이 정확히 일치함을 `assert`로 확인했고, rename 후 `stores_2021.columns.tolist() == stores_2025.columns.tolist()`가 `True`임을 확인한 뒤 5개 연도를 `concat`했다.
2. **sales 시간대별 매출 건수 컬럼명 처리**: `시간대_건수~06_매출_건수` 등 6개 컬럼은 같은 위치의 `시간대_00~06_매출_금액` 등 금액 컬럼과 순서·구간이 정확히 대응되는 것을 확인하여, `TIME_COUNT_RENAME_MAP` 명시적 dictionary로 `시간대_00~06_매출_건수` ~ `시간대_21~24_매출_건수`로 정리했다(rename 전후 컬럼 수 55개로 동일, 중복 컬럼 없음 확인).
3. **데이터 타입 통일**: `기준_년분기_코드`, `상권_코드`는 세 데이터 모두 `int64`로 명시 변환(변환 전후 고유값·결측치 동일함을 검증)했다. `상권_구분_코드`, `상권_구분_코드_명`, `상권_코드_명`, `서비스_업종_코드`, `서비스_업종_코드_명`은 애초에 숫자로만 구성되지 않아(예: `A`/`D`/`R`/`U`, `CS100001`) pandas가 이미 문자열 계열(`str`) dtype으로 읽었으며, 이를 그대로 유지해 앞자리 손실 없이 문자열로 보존했다. 최종적으로 세 데이터의 `상권_코드` dtype(`int64`)과 sales/stores의 `서비스_업종_코드` dtype(`str`)이 각각 일치함을 확인했다.
4. **key 식별자 무결성 및 sales/stores ↔ population 미매칭 key**: `기준_년분기_코드`는 20211~20254 범위와 연도(2021~2025)/분기(1~4) 논리적 유효성을 모두 만족한다. `sales -> stores` key(기준년분기+상권코드+서비스업종) 매칭률은 **100.00%**로 `01`과 동일하게 완전히 유지되었다. 반면 `(기준_년분기_코드, 상권_코드)` 기준으로는 sales(31,415개) < population(32,984개) < stores(33,000개) 순으로 수록 범위가 서로 완전히 같지 않아, sales/stores에만 있고 population에는 없는 key가 각 16건(3110946 청계산원터골·3110948 헌인가구단지·3111002 한국교통안전공단 강남자동차검사소의 일부 분기), 반대로 population에만 있고 sales에는 없는 key가 1,585건(청운초등학교 등 소규모 지점 다수) 존재함을 실제 key 목록으로 확인했다(섹션 17-1). 이 미매칭 행은 삭제하거나 population 값을 임의로 채우지 않고 사실만 기록했다.
5. **상권코드 ↔ 상권명 표기 불일치**: `상권_코드` 3110024가 sales/stores/population에서 `혜회동주민센터`·`혜화동주민센터` 두 표기로, population에서는 추가로 `상권_코드` 3110379가 `KT&G 북부지사`·`KTNG 북부지사` 두 표기로 연결되는 사례가 발견되었다. 현재로서는 공식 명칭을 확정할 근거가 없어 표기를 통일하거나 canonical-name 컬럼을 만들지 않았으며, 향후 분석에서는 `상권_코드`를 기본 식별자로, `상권_코드_명`은 표시용 속성으로만 사용하기로 방침을 정리했다(섹션 10-1). `서비스_업종_코드` ↔ `서비스_업종_코드_명`은 sales/stores 모두 불일치 사례 없음.
6. **결측치 처리**: sales/stores/population 모두 전체 결측치 0건, 핵심 key 컬럼(`기준_년분기_코드`, `상권_코드`, `서비스_업종_코드`) 결측도 0건이었다. 결측이 없었으므로 별도의 대체(imputation) 작업은 수행하지 않았다.
7. **중복 데이터**: grain key(sales/stores: 기준년분기+상권코드+서비스업종, population: 기준년분기+상권코드) 기준 중복, 완전 동일 행(exact duplicate) 모두 세 데이터에서 0건으로 확인되어 별도 제거 작업은 필요하지 않았다.
8. **비정상적인 값**: 매출 금액/건수, 점포 수 계열, 유동인구 계열 컬럼에서 음수 값은 전혀 발견되지 않았다. 다만 stores의 `개업_율`이 100을 초과하는 행이 14건, `폐업_률`이 100을 초과하는 행이 723건 있었다(0은 정상값으로 간주하고 삭제하지 않았으며, 100 초과 값도 삭제하지 않고 "확인 필요" 사항으로만 기록했다).
9. **구성 합계 검증**: sales의 시간대별 매출금액/건수 합은 당월 합계와 매우 높은 수준으로 일치했다(금액 100.00%, 건수 100.00% 완전 일치, 최대 절대 차이도 건수 기준 1 이내). 반면 성별(남성+여성) 및 연령대별 매출금액/건수 합은 당월 합계와 완전히 일치하는 비율이 약 41%에 그쳤고, 최대 절대 차이가 금액 기준 약 4,642억 원에 달하는 행도 있었다. 이 차이의 정확한 원인(집계 기준, 산출 방식, 추정·비식별화 처리 등)은 현재 데이터만으로 확정할 수 없어 값을 임의로 맞추지 않고 사실만 기록했으며, 향후 연령대 비율형 파생지표를 만들기 전 서울시 상권분석서비스의 공식 컬럼 정의를 추가로 확인할 필요가 있다는 점을 남겨두었다(섹션 14). population은 성별/연령대별/시간대별 합 모두 완전 일치 비율이 30~50% 수준이었지만 절대 차이의 중앙값이 0~1명, 최대 절대 차이도 5명 이내로 작았다.
10. **2021Q1~2025Q4 전체 기간 유지**: sales/stores/population 모두 20개 분기가 누락 없이 유지됨을 재확인했다.
11. **서울 전체 상권 데이터 유지**: 황학동 4개 상권으로 필터링하지 않았으며, 전체 기간 기준 고유 상권코드 수는 sales 1,596개, stores/population 각 1,650개로 서울 전체 상권이 그대로 보존되었다.
12. **황학 4개 상권 유지**: 3110055, 3110057, 3130054, 3130055 네 상권 모두 sales/stores/population 세 데이터에서 20개 분기 전체에 걸쳐 빠짐없이 존재함을 재확인했다(섹션 16의 검증표 참고). 이는 필터링이 아니라 손실 여부 확인용이며, 저장된 데이터는 서울 전체 상권을 포함한다.
13. **processed 저장 후 무결성**: 필수 분석 준비도 검증(섹션 18) 12개 항목이 모두 통과한 뒤 저장했고, 저장 후 재로드하여 shape, 컬럼명/순서, 핵심 key(`기준_년분기_코드`/`상권_코드`/`서비스_업종_코드` 등) dtype, 컬럼별 unique 개수, 그리고 grain key 조합 자체의 unique 개수까지 저장 전후 동일함을 `assert`로 검증했다(섹션 19, "저장 후 무결성 검증 통과" 출력 확인).

### 다음 단계 (`03_eda.ipynb` 예고)

이번 노트북을 통해 `data/processed/sales.csv`, `stores.csv`, `population.csv`는 2021Q1~2025Q4 전체 기간과 서울 전체 상권을 유지하면서도, key 정합성·dtype·결측치·중복·저장 무결성이 모두 검증된 상태로 정리되었다. 다음 `03_eda.ipynb`에서는 이 데이터를 불러와 황학동 4개 상권을 중심으로 매출·점포·생활인구의 시계열 변화와 업종별 변화를 살펴보고, 시간대·요일·연령대별 소비/인구 구조를 확인하며, 서울 전체 상권 분포 대비 황학동의 상대적 위치를 비교하여 이후 쇠퇴·회복 신호 지표를 설계하기 위한 기초 탐색을 진행할 예정이다. 이 데이터는 이후 황학동 vs 서울 전체 비교, 유사 상권 탐색, 상권 쇠퇴·회복 신호 설계, Tableau 대시보드, AI 기반 상권 진단의 기반으로 사용할 수 있는 상태이며, 이 노트북에서는 그 분석 자체를 다루지 않는다.
